## Supporting notebook to prepare hydrodynamic data for notebook 0308

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import opentnsim.fis as fis
import xarray as xr
import networkx as nx
from shapely.geometry import Point
import geopandas as gpd
from opentnsim.environment.utils import (create_default_hydrodynamic_dataset, 
                                         add_specific_environmental_data_on_node, 
                                         interpolate_data_on_route,
                                         add_closest_node_to_xr_dataset)
from opentnsim.graph.utils import get_closest_node_to_geometry
%matplotlib inline

#### Loading raw data

In [ ]:
path = os.getcwd()

In [ ]:
df = pd.read_csv(os.path.join(path,"raw_hydrodynamic_data_PoR.csv"), delimiter=';', encoding='ISO-8859-1', on_bad_lines='skip')

#### Preparing time data

In [ ]:
df['TIME'] = df['WAARNEMINGDATUM'] + ' ' + df['WAARNEMINGTIJD (MET/CET)']
df['TIME'] = pd.to_datetime(df['TIME'], format = "%d-%m-%Y %H:%M:%S")

#### Preparing location data

In [ ]:
df['X'] = df['X'].apply(lambda x: x.replace(",", "."))
df['Y'] = df['Y'].apply(lambda x: x.replace(",", "."))

#### Selecting columns

In [ ]:
df = df[['TIME','MEETPUNT_IDENTIFICATIE','NUMERIEKEWAARDE','X','Y','EPSG']]

#### Translating data

In [ ]:
df = df.rename(columns={'MEETPUNT_IDENTIFICATIE':'Location','NUMERIEKEWAARDE':'Water level'})

#### Indexing data

In [ ]:
df = df.set_index('TIME')

#### Selecting locations

In [ ]:
df_sel = df[df['Location'].isin(['Hoek van Holland','Geulhaven Radarpost 10',])]

#### Preparing water level data

In [ ]:
#cm to m
df_sel['Water level'] = df_sel['Water level']/100

# Interpolate erroneous data
df_sel.loc[df_sel[df_sel['Water level'] > 100].index,'Water level'] = np.nan
df_sel['Water level'] = df_sel['Water level'].interpolate()

#### Renaming station

In [ ]:
df_sel = df_sel.copy()
df_sel.loc[df_sel['Location'] == 'Geulhaven Radarpost 10', 'Location'] = 'Geulhaven'

#### Creating xr.DataSet

In [ ]:
hydrodynamic_data = xr.Dataset()

In [ ]:
# Creating numpy.array with water level data for each STATION and TIME + coordinates
stations = []
xs = []
ys = []
epsgs = []
wlev_data = []
for station, df_location in df_sel.groupby('Location'):
    x = df_location['X'].mode().iloc[0]
    y = df_location['Y'].mode().iloc[0]
    epsg = df_location['EPSG'].mode().iloc[0]
    time = df_location.index
    stations.append(station)
    xs.append(x)
    ys.append(y)
    epsgs.append(epsg)
    wlev_data.append(df_location['Water level'].values)
wlev_data = np.array(wlev_data)

# Creating dataarray
wlev_xr_da = xr.DataArray(wlev_data, 
                          dims = ('STATION','TIME'), 
                          coords={'STATION':stations,
                                  'TIME':time,
                                  'X': ("STATION", xs),
                                  'Y': ("STATION", ys),
                                  'EPSG': ("STATION", epsgs)})

# Adding dataarray to dataset
hydrodynamic_data['Water level'] = wlev_xr_da

# Saving data
hydrodynamic_data.to_netcdf('wlev_measurement_data_PoR.nc')